<a href="https://colab.research.google.com/github/s-shrikanth/sparksense/blob/main/Copy_of_Sparksense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# Load your uploaded files (names as shown in Colab)
data1 = pd.read_csv('CIA-1 Dataset - Dataset (1).csv')
data2 = pd.read_csv('ai4i2020.csv')

# Harmonize columns from ai4i2020 to match the required features
data2 = data2.rename(columns={
    'Air temperature [K]': 'Air temperature [K]',
    'Process temperature [K]': 'Process temperature [K]',
    'Rotational speed [rpm]': 'Rotational speed [rpm]',
    'Torque [Nm]': 'Torque [Nm]'
})
# For vibration, use 'Tool wear [min]' as proxy; create 'Operational Hours' as dummy if not present
if 'Vibration Levels ' not in data2.columns:
    data2['Vibration Levels '] = data2['Tool wear [min]']
if 'Operational Hours' not in data2.columns:
    data2['Operational Hours'] = 0  # You can update with realistic value if available

# Create a unified failure label for ai4i2020 (if not present)
if 'Failure Type' not in data2.columns:
    data2['Failure Type'] = np.where(data2['Machine failure'] == 1, 'Failure', 'No Failure')

# Concatenate both datasets
data = pd.concat([data1, data2], ignore_index=True)

# Select features
features = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]',
            'Torque [Nm]', 'Vibration Levels ', 'Operational Hours']
X = data[features]

# Prepare target variable
y_fault = data['Failure Type']

# Encode labels
label_encoder = LabelEncoder()
y_fault_encoded = label_encoder.fit_transform(y_fault)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data for classification
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_scaled, y_fault_encoded, test_size=0.3, random_state=42)

# Stage 1: Isolation Forest on normal data only
X_normal = X_scaled[y_fault == 'No Failure']
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X_normal)

# Predict anomalies in test set
anomaly_pred = iso_forest.predict(X_test_cls)  # -1: anomaly, 1: normal

# Extract anomalies for classification
anomalies_idx = np.where(anomaly_pred == -1)[0]
X_test_anomalies = X_test_cls[anomalies_idx]
y_test_anomalies = y_test_cls[anomalies_idx]

# Stage 2: Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_cls, y_train_cls)

if len(X_test_anomalies) > 0:
    fault_pred = clf.predict(X_test_anomalies)
    fault_pred_labels = label_encoder.inverse_transform(fault_pred)
    y_true_labels = label_encoder.inverse_transform(y_test_anomalies)

    print("Classification Report for Faults Detected:")
    print(classification_report(y_true_labels, fault_pred_labels))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true_labels, fault_pred_labels))

    print("Accuracy:", accuracy_score(y_true_labels, fault_pred_labels))
else:
    print("No anomalies detected in the test set for classification.")

# Save models (you can download these later if needed)
joblib.dump(iso_forest, 'isolation_forest_model.joblib')
joblib.dump(clf, 'random_forest_classifier.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(label_encoder, 'label_encoder.joblib')


Classification Report for Faults Detected:
                    precision    recall  f1-score   support

           Failure       0.80      0.73      0.76        22
        No Failure       0.91      0.98      0.94       167
Overstrain Failure       0.00      0.00      0.00         1
     Power Failure       0.00      0.00      0.00         3
 Tool Wear Failure       0.00      0.00      0.00         6

          accuracy                           0.90       199
         macro avg       0.34      0.34      0.34       199
      weighted avg       0.85      0.90      0.87       199

Confusion Matrix:
[[ 16   6   0   0   0]
 [  4 163   0   0   0]
 [  0   1   0   0   0]
 [  0   3   0   0   0]
 [  0   6   0   0   0]]
Accuracy: 0.8994974874371859


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


['label_encoder.joblib']

In [ ]:
import pandas as pd
import numpy as np
import glob
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# --- Load ALL dataset CSVs you have uploaded ---
csv_files = glob.glob('*.csv')
print(f"Loading these datasets: {csv_files}")

# Read and concatenate data
all_dfs = []
for filename in csv_files:
    df = pd.read_csv(filename)
    # Harmonize for main features and labels
    if 'Tool wear [min]' in df.columns:
        df['Vibration Levels '] = df.get('Vibration Levels ', df['Tool wear [min]'])
    if 'Operational Hours' not in df.columns:
        df['Operational Hours'] = df.get('Operational Hours', 0)
    if 'Failure Type' not in df.columns and 'Machine failure' in df.columns:
        df['Failure Type'] = np.where(df['Machine failure']==1, 'Failure', 'No Failure')
    all_dfs.append(df)

data = pd.concat(all_dfs, ignore_index=True)

# --- Define features and target ---
main_features = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]',
    'Vibration Levels ', 'Operational Hours'
]
X = data[main_features]
y_fault = data['Failure Type']

# --- Encode target labels ---
label_encoder = LabelEncoder()
y_fault_encoded = label_encoder.fit_transform(y_fault)

# --- Scale features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Train/test split for classification ---
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_scaled, y_fault_encoded, test_size=0.3, random_state=42)

# --- Isolation Forest (anomaly prediction) on "No Failure" samples ---
X_normal = X_scaled[y_fault == 'No Failure']
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X_normal)

# --- Example machine ratings (customize as needed) ---
machine_ratings = {
    'Air temperature [K]': 320,
    'Process temperature [K]': 315,
    'Rotational speed [rpm]': 2000,
    'Torque [Nm]': 60,
    'Vibration Levels ': 60,
    'Operational Hours': 1000
}

def check_threshold_warnings(readings):
    """Check if any reading is near rating limit and print warnings."""
    for param, max_val in machine_ratings.items():
        value = readings.get(param, None)
        if value is not None and value > 0.95 * max_val:
            print(f"Warning: {param} at {value:.2f} is near its max rating {max_val}")

# --- Predict anomalies & early warning on near-threshold values ---
anomaly_pred = iso_forest.predict(X_test_cls)  # -1 anomaly, 1 normal
anomalies_idx = np.where(anomaly_pred == -1)[0]
X_test_anomalies = X_test_cls[anomalies_idx]
y_test_anomalies = y_test_cls[anomalies_idx]

if len(X_test_anomalies) > 0:
    anomalies_original = scaler.inverse_transform(X_test_anomalies)
    for i, sample in enumerate(anomalies_original, 1):
        print(f"\nWARNING CHECK - Anomaly Sample #{i} sensor readings:")
        reading = dict(zip(main_features, sample))
        check_threshold_warnings(reading)

# --- Train Fault Classifier ---
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_cls, y_train_cls)

# --- Classify detected anomalies ---
if len(X_test_anomalies) > 0:
    fault_pred = clf.predict(X_test_anomalies)
    fault_pred_labels = label_encoder.inverse_transform(fault_pred)
    y_true_labels = label_encoder.inverse_transform(y_test_anomalies)

    solution_mapper = {
        "Tool Wear Failure": "Inspect the tool for wear; replace if needed.",
        "Power Failure": "Verify power supply, circuit breakers, and wiring.",
        "Overstrain Failure": "Reduce load; check for mechanical stresses or misalignment.",
        "Failure": "General failure detected; consult diagnostics and maintenance logs.",
        "No Failure": "System operating normally, no action required."
    }

    print("\nFAULT CLASSIFICATION & RECOMMENDED SOLUTIONS:")
    from collections import Counter
    counts = Counter(fault_pred_labels)
    for fault, count in counts.items():
        solution = solution_mapper.get(fault, "Consult maintenance team for guidance.")
        print(f"- {fault} (Count: {count}): {solution}")

    print("\nClassification Report:")
    print(classification_report(y_true_labels, fault_pred_labels))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true_labels, fault_pred_labels))
    print(f"Overall Accuracy: {accuracy_score(y_true_labels, fault_pred_labels):.3f}")
else:
    print("No anomalies detected in test set for classification.")

# --- Save models and encoders for later use ---
joblib.dump(iso_forest, 'isolation_forest_model.joblib')
joblib.dump(clf, 'random_forest_classifier.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(label_encoder, 'label_encoder.joblib')


Loading these datasets: ['predictive_maintenance_dataset.csv', 'CIA-1 Dataset - Dataset (1).csv', 'classData.csv', 'ai4i2020.csv', 'predictive_maintenance.csv']

WARNING CHECK - Anomaly Sample #1 sensor readings:

WARNING CHECK - Anomaly Sample #2 sensor readings:

WARNING CHECK - Anomaly Sample #3 sensor readings:

WARNING CHECK - Anomaly Sample #4 sensor readings:

WARNING CHECK - Anomaly Sample #5 sensor readings:

WARNING CHECK - Anomaly Sample #6 sensor readings:

WARNING CHECK - Anomaly Sample #7 sensor readings:

WARNING CHECK - Anomaly Sample #8 sensor readings:

WARNING CHECK - Anomaly Sample #9 sensor readings:

WARNING CHECK - Anomaly Sample #10 sensor readings:

WARNING CHECK - Anomaly Sample #11 sensor readings:

WARNING CHECK - Anomaly Sample #12 sensor readings:

WARNING CHECK - Anomaly Sample #13 sensor readings:

WARNING CHECK - Anomaly Sample #14 sensor readings:

WARNING CHECK - Anomaly Sample #15 sensor readings:

WARNING CHECK - Anomaly Sample #16 sensor readings:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


['label_encoder.joblib']